# Truthfulness Comparison Fine-tuning
## This notebook fine-tunes a language model to predict which answer is more truthful

In [1]:
# %% 
# Install required packages (run once)  
!pip install --quiet transformers datasets torch pandas scikit-learn tqdm matplotlib seaborn accelerate


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Trainer will use all available GPUs by default if you have multiple
# Just make sure CUDA_VISIBLE_DEVICES isn't limiting you
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Use GPUs 0 and 1 only
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

CUDA_VISIBLE_DEVICES: 0,1,2,3,4,5,6,7


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"

PROMPT = """Human: Please suggest a few papers to consider based on the search term given. The names of the papers should be listed.\n\nTopic: scaling law + machine learning
    Response 1: 1. \"On the Powerlaw Distribution in Machine Learning\" by S.D. Kolaczyk and J. M. Landwehr.\n2. \"The Muth's Law and Application to Machine Learning\" by D.A. Muth and L.V. Prokopenko.\n3. \"The Powerlaw Distribution in Machine Learning\" by H. Liu and B. Liu.
    Response 2: 1. \"A Scaling Law for Machine Learning Algorithms on Multicore and Manycore Architectures\" by Yingfei Xiong et al. \r\n2. \"Scaling Laws for Machine Learning Algorithms\" by Suvrit Sra et al. \r\n3. \"Scaling Laws for Machine Learning on Multicore Architectures\" by Richard Yoo et al. \r\n4. \"A Scaling Law for Machine Learning on Multicore and Manycore Architectures\" by Yuriy Brun et al.
    Claim: Response 2 is more helpful and harmless than Response 1.
    I think this claim is """

REVERSE_PROMPT = """Human: Please suggest a few papers to consider based on the search term given. The names of the papers should be listed.\n\nTopic: scaling law + machine learning
    Response 1: 1. \"A Scaling Law for Machine Learning Algorithms on Multicore and Manycore Architectures\" by Yingfei Xiong et al. \r\n2. \"Scaling Laws for Machine Learning Algorithms\" by Suvrit Sra et al. \r\n3. \"Scaling Laws for Machine Learning on Multicore Architectures\" by Richard Yoo et al. \r\n4. \"A Scaling Law for Machine Learning on Multicore and Manycore Architectures\" by Yuriy Brun et al.
    Response 2: 1. \"On the Powerlaw Distribution in Machine Learning\" by S.D. Kolaczyk and J. M. Landwehr.\n2. \"The Muth's Law and Application to Machine Learning\" by D.A. Muth and L.V. Prokopenko.\n3. \"The Powerlaw Distribution in Machine Learning\" by H. Liu and B. Liu.
    Claim: Response 2 is more helpful and harmless than Response 1.
    I think this claim is """

# ── SETUP ──────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── TOKENIZE ───────────────────────────────────────────────────────────────────
inputs = tokenizer(PROMPT, return_tensors="pt")
reverse_inputs = tokenizer(REVERSE_PROMPT, return_tensors="pt")

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

# Zero Shot Finetuning

In [ ]:
zero shot for 64, at every epoch

In [ ]:


import random
random.seed(42)
import json
from collections import defaultdict
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from accelerate import Accelerator

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME   = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b"
TRAIN_PATH   = "data/train_gsm8k_preference.json"
TEST_PATH    = "data/test_gsm8k_preference.json"
TRAIN_SUBSET = 11664               # how many original GSM8K items to load
LR           = 5e-5                # learning rate
RANK         = 128                 # LoRA rank
BATCH_PAIRS  = 16                  # examples per optimizer step
# NUM_EPOCHS   = 3                 # how many passes over train set

# ── SETUP ACCELERATOR ──────────────────────────────────────────────────────────
accelerator = Accelerator(mixed_precision="bf16")
device      = accelerator.device

# ── TOKENIZER & BASE MODEL ────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

# if no pad token, use eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── WRAP IN LoRA FOR EFFICIENT FINETUNING ─────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=RANK,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)

# ── WHICH TOKENS CORRESPOND TO “ True” vs. “ False” ───────────────────────────
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]

# ── PROMPT TEMPLATE (unchanged) ───────────────────────────────────────────────
template = (
    "Question: {question}\n"
    "Claim: {answer}\n"
    "I think this claim is"
)

# ── BUILD TRAINING EXAMPLES ───────────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    raw_train = json.load(f)
# restrict to first TRAIN_SUBSET items
train_data = raw_train[:TRAIN_SUBSET]

# train_examples = []
# for ex in train_data:
#     # format the prompt using the 'choice' field as the claimed answer
#     prompt = template.format(
#         question=ex["question"],
#         answer=ex["choice"]
#     )
#     # map the boolean/string label → integer
#     # if ex["label"] is already a Python bool, this works directly;
#     # if it's a string, you can do ex["label"].lower() == "true"
#     lbl = 1 if ex["label"] else 0

#     train_examples.append((prompt, lbl))

# train_examples = []
# # assume train_data is already ordered so that every two entries share the same question
# for i in range(0, len(train_data), 2):
#     ex1, ex2 = train_data[i], train_data[i+1]
#     assert ex1["question"] == ex2["question"], f"Pair mismatch at indices {i},{i+1}"

#     q = ex1["question"]
#     # build the two prompts
#     p1 = template.format(question=q, answer=ex1["choice"])
#     p2 = template.format(question=q, answer=ex2["choice"])

#     # pick a random bit once per pair, then complement it
#     bit   = random.randint(0, 1)   # either 0 or 1
#     lbl1  = bit
#     lbl2  = 1 - bit                # guaranteed opposite of lbl1

#     train_examples.append((p1, lbl1))
#     train_examples.append((p2, lbl2))

# ── 1) LOAD & GROUP TRAIN JSON BY ID ─────────────────────────────────────────
# with open(TRAIN_PATH) as f:
#     raw = json.load(f)

groups = defaultdict(list)
for ex in train_data:
    groups[ex["consistency_id"]].append(ex)

# ── 2) BUILD LIST OF (prompt1, prompt2) PAIRS ─────────────────────────────────
train_pairs = []
for cid, examples in groups.items():
    assert len(examples)==2, f"ID {cid} has {len(examples)} examples!"
    e1, e2 = examples
    q = e1["question"]
    p1 = template.format(question=q, answer=e1["choice"])
    p2 = template.format(question=q, answer=e2["choice"])
    train_pairs.append((p1, p2))

# ── 3) DATALOADER THAT YIELDS BATCHES OF PAIRS ────────────────────────────────
def collate_pairs(batch_pairs):
    # batch_pairs: list of (p1,p2)
    flat = []
    for p1,p2 in batch_pairs:
        flat.extend([p1,p2])
    return flat   # length = 2 * BATCH_PAIRS

# DataLoader will yield batches of (prompt, label) tuples
train_loader = DataLoader(
    train_pairs,
    batch_size=BATCH_PAIRS,
    shuffle=True,        # shuffle at the PAIR level
    collate_fn=collate_pairs
)

# ── BUILD TEST EXAMPLES ────────────────────────────────────────────────────────
with open(TEST_PATH) as f:
    test_data = json.load(f)

test_examples = []
for ex in test_data:
    # format the prompt using the 'choice' field as the claimed answer
    prompt = template.format(
        question=ex["question"],
        answer=ex["choice"]
    )
    # map the boolean/string label → integer
    # if ex["label"] is already a Python bool, this works directly;
    # if it's a string, you can do ex["label"].lower() == "true"
    lbl = 1 if ex["label"] else 0

    test_examples.append((prompt, lbl))

# ── PREPARE MODEL & OPTIMIZER ──────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

model.eval()
correct = 0
total   = 0
epoch = 1
step_count   = 0   # how many optimizer.step() calls
sample_count = 0   # how many examples processed
step = 0
while True:
    model.train()
    total_loss = 0.0
    for prompts_flat in tqdm(train_loader, desc="Training"):
        # prompts_flat is a list of 2*BATCH_PAIRS strings
        sample_count += len(prompts_flat)
        enc = tokenizer(
            prompts_flat,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(device)

        # forward
        logits = model(**enc).logits  # [2*BATCH_PAIRS, seq_len, vocab]
        lens   = enc["attention_mask"].sum(dim=1)
        last   = logits[torch.arange(len(prompts_flat)), lens-1]  # [2*B, vocab]

        # extract the two logits of interest
        l_false = last[:, false_id]  # [2*BATCH_PAIRS]
        l_true  = last[:, true_id]   # [2*BATCH_PAIRS]

        # compute margins and regroup into pairs
        margins = (l_true - l_false).detach().cpu().view(-1, 2)  # [BATCH_PAIRS, 2]

        # build pseudo labels: [1,0] if first margin > second, else [0,1]
        pseudo = sum(
            ([1, 0] if m[0] > m[1] else [0, 1] for m in margins),
            []
        )
        gold = torch.tensor(pseudo, dtype=torch.long, device=device)  # [2*BATCH_PAIRS]

        # loss & step
        pair_logits = torch.stack([l_false, l_true], dim=1)  # [2*B,2]
        loss = F.cross_entropy(pair_logits, gold)
        accelerator.backward(loss)
        step_count += 1
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    print(f"\nEpoch {epoch} avg training loss: {total_loss/len(train_loader):.4f}")
    print(f"  ↳ gradient steps: {step_count}")
    print(f"  ↳ samples seen:   {sample_count}")
    # ---- EVALUATION PASS ----
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for prompt, label in tqdm(test_examples, desc=f"Epoch {epoch} Eval"):
            # reuse your score_text helper if you like,
            # but here we'll inline for clarity:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
            logits = model(**enc).logits
            length = enc["attention_mask"].sum(dim=1  )
            last   = logits[0, length-1]
            logp   = torch.log_softmax(last, dim=-1)    
            # decision = log P(True) - log P(False)
            if (logp[:, true_id         ] - logp[:, false_id]).item() >= 0:
                pred = 1
            else:
                pred = 0
            correct += (pred == label)
            total   += 1

    acc = correct/total
    print(f"Epoch {epoch} → Eval accuracy: {acc:.4f}\n")
    epoch += 1

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Training: 100%|██████████| 359/359 [07:13<00:00,  1.21s/it]



Epoch 1 avg training loss: 0.3995
  ↳ gradient steps: 359
  ↳ samples seen:   11462


Epoch 1 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.27it/s]


Epoch 1 → Eval accuracy: 0.8028



...

Training: 100%|██████████| 359/359 [07:15<00:00,  1.21s/it]



Epoch 169 avg training loss: 0.0000
  ↳ gradient steps: 60671
  ↳ samples seen:   1937078


Epoch 169 Eval: 100%|██████████| 1486/1486 [01:18<00:00, 18.95it/s]


Epoch 169 → Eval accuracy: 0.8318



Training:  21%|██        | 74/359 [01:31<07:02,  1.48s/it]

## 4096

In [ ]:
Training: 100%|██████████| 128/128 [02:49<00:00,  1.33s/it]

Epoch 1 avg training loss: 0.4497
  ↳ gradient steps: 128
  ↳ samples seen:   4096
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:13<00:00, 20.22it/s]
Epoch 1 → Eval accuracy: 0.7214

Training: 100%|██████████| 128/128 [02:44<00:00,  1.29s/it]

Epoch 2 avg training loss: 0.3437
  ↳ gradient steps: 256
  ↳ samples seen:   8192
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.39it/s]
Epoch 2 → Eval accuracy: 0.7645

Training: 100%|██████████| 128/128 [02:41<00:00,  1.26s/it]

Epoch 3 avg training loss: 0.2695
  ↳ gradient steps: 384
  ↳ samples seen:   12288
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.77it/s]
Epoch 3 → Eval accuracy: 0.7813

Training: 100%|██████████| 128/128 [02:45<00:00,  1.29s/it]

Epoch 4 avg training loss: 0.2073
  ↳ gradient steps: 512
  ↳ samples seen:   16384
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:05<00:00, 22.73it/s]
Epoch 4 → Eval accuracy: 0.7450

Training: 100%|██████████| 128/128 [02:49<00:00,  1.33s/it]

Epoch 5 avg training loss: 0.1356
  ↳ gradient steps: 640
  ↳ samples seen:   20480
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.05it/s]
Epoch 5 → Eval accuracy: 0.7658

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 6 avg training loss: 0.0888
  ↳ gradient steps: 768
  ↳ samples seen:   24576
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:02<00:00, 23.95it/s]
Epoch 6 → Eval accuracy: 0.7699

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 7 avg training loss: 0.0606
  ↳ gradient steps: 896
  ↳ samples seen:   28672
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.23it/s]
Epoch 7 → Eval accuracy: 0.7564

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 8 avg training loss: 0.0485
  ↳ gradient steps: 1024
  ↳ samples seen:   32768
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:05<00:00, 22.83it/s]
Epoch 8 → Eval accuracy: 0.7584

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 9 avg training loss: 0.0499
  ↳ gradient steps: 1152
  ↳ samples seen:   36864
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.12it/s]
Epoch 9 → Eval accuracy: 0.7921

Training: 100%|██████████| 128/128 [02:52<00:00,  1.34s/it]

Epoch 10 avg training loss: 0.0321
  ↳ gradient steps: 1280
  ↳ samples seen:   40960
Epoch 10 Eval: 100%|██████████| 1486/1486 [01:17<00:00, 19.09it/s]
Epoch 10 → Eval accuracy: 0.7295

Training: 100%|██████████| 128/128 [02:45<00:00,  1.30s/it]

Epoch 11 avg training loss: 0.0219
  ↳ gradient steps: 1408
  ↳ samples seen:   45056
Epoch 11 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.09it/s]
Epoch 11 → Eval accuracy: 0.7712

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 12 avg training loss: 0.0164
  ↳ gradient steps: 1536
  ↳ samples seen:   49152
Epoch 12 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.08it/s]
Epoch 12 → Eval accuracy: 0.7187

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 13 avg training loss: 0.0244
  ↳ gradient steps: 1664
  ↳ samples seen:   53248
Epoch 13 Eval: 100%|██████████| 1486/1486 [01:13<00:00, 20.23it/s]
Epoch 13 → Eval accuracy: 0.7517

Training: 100%|██████████| 128/128 [02:49<00:00,  1.32s/it]

Epoch 14 avg training loss: 0.0191
  ↳ gradient steps: 1792
  ↳ samples seen:   57344
Epoch 14 Eval: 100%|██████████| 1486/1486 [01:17<00:00, 19.18it/s]
Epoch 14 → Eval accuracy: 0.7604

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 15 avg training loss: 0.0118
  ↳ gradient steps: 1920
  ↳ samples seen:   61440
Epoch 15 Eval: 100%|██████████| 1486/1486 [01:04<00:00, 22.92it/s]
Epoch 15 → Eval accuracy: 0.7725

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 16 avg training loss: 0.0043
  ↳ gradient steps: 2048
  ↳ samples seen:   65536
Epoch 16 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.32it/s]
Epoch 16 → Eval accuracy: 0.7793

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 17 avg training loss: 0.0078
  ↳ gradient steps: 2176
  ↳ samples seen:   69632
Epoch 17 Eval: 100%|██████████| 1486/1486 [00:55<00:00, 26.75it/s]
Epoch 17 → Eval accuracy: 0.7692

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 18 avg training loss: 0.0222
  ↳ gradient steps: 2304
  ↳ samples seen:   73728
Epoch 18 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.02it/s]
Epoch 18 → Eval accuracy: 0.7880

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 19 avg training loss: 0.0146
  ↳ gradient steps: 2432
  ↳ samples seen:   77824
Epoch 19 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 17.98it/s]
Epoch 19 → Eval accuracy: 0.7840

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 20 avg training loss: 0.0166
  ↳ gradient steps: 2560
  ↳ samples seen:   81920
Epoch 20 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.32it/s]
Epoch 20 → Eval accuracy: 0.7685

Training: 100%|██████████| 128/128 [02:49<00:00,  1.32s/it]

Epoch 21 avg training loss: 0.0094
  ↳ gradient steps: 2688
  ↳ samples seen:   86016
Epoch 21 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.68it/s]
Epoch 21 → Eval accuracy: 0.7645

Training: 100%|██████████| 128/128 [02:44<00:00,  1.29s/it]

Epoch 22 avg training loss: 0.0098
  ↳ gradient steps: 2816
  ↳ samples seen:   90112
Epoch 22 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.03it/s]
Epoch 22 → Eval accuracy: 0.7651

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 23 avg training loss: 0.0142
  ↳ gradient steps: 2944
  ↳ samples seen:   94208
Epoch 23 Eval: 100%|██████████| 1486/1486 [01:05<00:00, 22.81it/s]
Epoch 23 → Eval accuracy: 0.7618

Training: 100%|██████████| 128/128 [02:42<00:00,  1.27s/it]

Epoch 24 avg training loss: 0.0053
  ↳ gradient steps: 3072
  ↳ samples seen:   98304
Epoch 24 Eval: 100%|██████████| 1486/1486 [01:23<00:00, 17.84it/s]
Epoch 24 → Eval accuracy: 0.7732

Training: 100%|██████████| 128/128 [02:43<00:00,  1.28s/it]

Epoch 25 avg training loss: 0.0057
  ↳ gradient steps: 3200
  ↳ samples seen:   102400
Epoch 25 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.61it/s]
Epoch 25 → Eval accuracy: 0.7322

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 26 avg training loss: 0.0023
  ↳ gradient steps: 3328
  ↳ samples seen:   106496
Epoch 26 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.36it/s]
Epoch 26 → Eval accuracy: 0.7557

Training: 100%|██████████| 128/128 [02:44<00:00,  1.28s/it]

Epoch 27 avg training loss: 0.0043
  ↳ gradient steps: 3456
  ↳ samples seen:   110592
Epoch 27 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.60it/s]
Epoch 27 → Eval accuracy: 0.7678

Training: 100%|██████████| 128/128 [02:44<00:00,  1.29s/it]

Epoch 28 avg training loss: 0.0446
  ↳ gradient steps: 3584
  ↳ samples seen:   114688
Epoch 28 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.29it/s]
Epoch 28 → Eval accuracy: 0.7820

Training: 100%|██████████| 128/128 [02:42<00:00,  1.27s/it]

Epoch 29 avg training loss: 0.0146
  ↳ gradient steps: 3712
  ↳ samples seen:   118784
Epoch 29 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.00it/s]
Epoch 29 → Eval accuracy: 0.7510

Training: 100%|██████████| 128/128 [02:50<00:00,  1.33s/it]

Epoch 30 avg training loss: 0.0098
  ↳ gradient steps: 3840
  ↳ samples seen:   122880
Epoch 30 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.31it/s]
Epoch 30 → Eval accuracy: 0.7752

Training: 100%|██████████| 128/128 [02:45<00:00,  1.30s/it]

Epoch 31 avg training loss: 0.0096
  ↳ gradient steps: 3968
  ↳ samples seen:   126976
Epoch 31 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.30it/s]
Epoch 31 → Eval accuracy: 0.7752

Training: 100%|██████████| 128/128 [02:45<00:00,  1.29s/it]

Epoch 32 avg training loss: 0.0078
  ↳ gradient steps: 4096
  ↳ samples seen:   131072
Epoch 32 Eval: 100%|██████████| 1486/1486 [01:08<00:00, 21.85it/s]
Epoch 32 → Eval accuracy: 0.7719

Training: 100%|██████████| 128/128 [02:48<00:00,  1.31s/it]

Epoch 33 avg training loss: 0.0053
  ↳ gradient steps: 4224
  ↳ samples seen:   135168
Epoch 33 Eval: 100%|██████████| 1486/1486 [01:12<00:00, 20.40it/s]
Epoch 33 → Eval accuracy: 0.7201

Training: 100%|██████████| 128/128 [02:48<00:00,  1.32s/it]

Epoch 34 avg training loss: 0.0145
  ↳ gradient steps: 4352
  ↳ samples seen:   139264
Epoch 34 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.00it/s]
Epoch 34 → Eval accuracy: 0.7651

Training: 100%|██████████| 128/128 [02:51<00:00,  1.34s/it]

Epoch 35 avg training loss: 0.0110
  ↳ gradient steps: 4480
  ↳ samples seen:   143360
Epoch 35 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.22it/s]
Epoch 35 → Eval accuracy: 0.7954

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 36 avg training loss: 0.0075
  ↳ gradient steps: 4608
  ↳ samples seen:   147456
Epoch 36 Eval: 100%|██████████| 1486/1486 [01:23<00:00, 17.76it/s]
Epoch 36 → Eval accuracy: 0.7577

Training: 100%|██████████| 128/128 [02:42<00:00,  1.27s/it]

Epoch 37 avg training loss: 0.0071
  ↳ gradient steps: 4736
  ↳ samples seen:   151552
Epoch 37 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.29it/s]
Epoch 37 → Eval accuracy: 0.7759

Training: 100%|██████████| 128/128 [02:44<00:00,  1.28s/it]

Epoch 38 avg training loss: 0.0113
  ↳ gradient steps: 4864
  ↳ samples seen:   155648
Epoch 38 Eval: 100%|██████████| 1486/1486 [01:18<00:00, 19.01it/s]
Epoch 38 → Eval accuracy: 0.7840

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 39 avg training loss: 0.0028
  ↳ gradient steps: 4992
  ↳ samples seen:   159744
Epoch 39 Eval: 100%|██████████| 1486/1486 [01:03<00:00, 23.26it/s]
Epoch 39 → Eval accuracy: 0.7436

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 40 avg training loss: 0.0084
  ↳ gradient steps: 5120
  ↳ samples seen:   163840
Epoch 40 Eval: 100%|██████████| 1486/1486 [01:03<00:00, 23.36it/s]
Epoch 40 → Eval accuracy: 0.7564

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 41 avg training loss: 0.0072
  ↳ gradient steps: 5248
  ↳ samples seen:   167936
Epoch 41 Eval: 100%|██████████| 1486/1486 [01:23<00:00, 17.90it/s]
Epoch 41 → Eval accuracy: 0.7577

Training: 100%|██████████| 128/128 [02:48<00:00,  1.31s/it]

Epoch 42 avg training loss: 0.0176
  ↳ gradient steps: 5376
  ↳ samples seen:   172032
Epoch 42 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 17.96it/s]
Epoch 42 → Eval accuracy: 0.7813

Training: 100%|██████████| 128/128 [02:43<00:00,  1.28s/it]

Epoch 43 avg training loss: 0.0022
  ↳ gradient steps: 5504
  ↳ samples seen:   176128
Epoch 43 Eval: 100%|██████████| 1486/1486 [01:17<00:00, 19.13it/s]
Epoch 43 → Eval accuracy: 0.7739

Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 44 avg training loss: 0.0028
  ↳ gradient steps: 5632
  ↳ samples seen:   180224
Epoch 44 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.35it/s]
Epoch 44 → Eval accuracy: 0.7692

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 45 avg training loss: 0.0006
  ↳ gradient steps: 5760
  ↳ samples seen:   184320
Epoch 45 Eval: 100%|██████████| 1486/1486 [00:57<00:00, 25.77it/s]
Epoch 45 → Eval accuracy: 0.7692

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 46 avg training loss: 0.0065
  ↳ gradient steps: 5888
  ↳ samples seen:   188416
Epoch 46 Eval: 100%|██████████| 1486/1486 [01:08<00:00, 21.59it/s]
Epoch 46 → Eval accuracy: 0.7961

Training: 100%|██████████| 128/128 [02:46<00:00,  1.30s/it]

Epoch 47 avg training loss: 0.0239
  ↳ gradient steps: 6016
  ↳ samples seen:   192512

## 16384

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:03<00:00,  1.29it/s]
Training: 100%|██████████| 359/359 [07:17<00:00,  1.22s/it]

Epoch 1 avg training loss: 0.4073
  ↳ gradient steps: 359
  ↳ samples seen:   11462
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.71it/s]
Epoch 1 → Eval accuracy: 0.7638

Training: 100%|██████████| 359/359 [07:15<00:00,  1.21s/it]

Epoch 2 avg training loss: 0.3168
  ↳ gradient steps: 718
  ↳ samples seen:   22924
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:08<00:00, 21.84it/s]
Epoch 2 → Eval accuracy: 0.8163

Training: 100%|██████████| 359/359 [07:07<00:00,  1.19s/it]

Epoch 3 avg training loss: 0.2471
  ↳ gradient steps: 1077
  ↳ samples seen:   34386
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.39it/s]
Epoch 3 → Eval accuracy: 0.8022

Training: 100%|██████████| 359/359 [07:17<00:00,  1.22s/it]

Epoch 4 avg training loss: 0.1778
  ↳ gradient steps: 1436
  ↳ samples seen:   45848
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.12it/s]
Epoch 4 → Eval accuracy: 0.8156

Training: 100%|██████████| 359/359 [07:13<00:00,  1.21s/it]

Epoch 5 avg training loss: 0.1041
  ↳ gradient steps: 1795
  ↳ samples seen:   57310
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.27it/s]
Epoch 5 → Eval accuracy: 0.8244

Training: 100%|██████████| 359/359 [07:11<00:00,  1.20s/it]

Epoch 6 avg training loss: 0.0691
  ↳ gradient steps: 2154
  ↳ samples seen:   68772
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:23<00:00, 17.88it/s]
Epoch 6 → Eval accuracy: 0.8022

Training: 100%|██████████| 359/359 [07:17<00:00,  1.22s/it]

Epoch 7 avg training loss: 0.0393
  ↳ gradient steps: 2513
  ↳ samples seen:   80234
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.14it/s]
Epoch 7 → Eval accuracy: 0.8264

Training: 100%|██████████| 359/359 [07:14<00:00,  1.21s/it]

Epoch 8 avg training loss: 0.0308
  ↳ gradient steps: 2872
  ↳ samples seen:   91696
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.54it/s]
Epoch 8 → Eval accuracy: 0.8035

Training: 100%|██████████| 359/359 [07:10<00:00,  1.20s/it]

Epoch 9 avg training loss: 0.0215
  ↳ gradient steps: 3231
  ↳ samples seen:   103158
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:07<00:00, 21.91it/s]
Epoch 9 → Eval accuracy: 0.7806

Training: 100%|██████████| 359/359 [07:13<00:00,  1.21s/it]

Epoch 10 avg training loss: 0.0193
  ↳ gradient steps: 3590
  ↳ samples seen:   114620
Epoch 10 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.37it/s]
Epoch 10 → Eval accuracy: 0.8143

Training: 100%|██████████| 359/359 [07:09<00:00,  1.20s/it]

Epoch 11 avg training loss: 0.0162
  ↳ gradient steps: 3949
  ↳ samples seen:   126082
Epoch 11 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.07it/s]
Epoch 11 → Eval accuracy: 0.8129

Training: 100%|██████████| 359/359 [07:15<00:00,  1.21s/it]

Epoch 12 avg training loss: 0.0169
  ↳ gradient steps: 4308
  ↳ samples seen:   137544
Epoch 12 Eval: 100%|██████████| 1486/1486 [00:50<00:00, 29.44it/s]
Epoch 12 → Eval accuracy: 0.8109

Training: 100%|██████████| 359/359 [07:09<00:00,  1.20s/it]

Epoch 13 avg training loss: 0.0167
  ↳ gradient steps: 4667
  ↳ samples seen:   149006
Epoch 13 Eval: 100%|██████████| 1486/1486 [01:04<00:00, 23.15it/s]
Epoch 13 → Eval accuracy: 0.8311

Training: 100%|██████████| 359/359 [07:15<00:00,  1.21s/it]

Epoch 14 avg training loss: 0.0170
  ↳ gradient steps: 5026
  ↳ samples seen:   160468
Epoch 14 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.31it/s]
Epoch 14 → Eval accuracy: 0.8048

Training: 100%|██████████| 359/359 [07:12<00:00,  1.21s/it]

Epoch 15 avg training loss: 0.0164
  ↳ gradient steps: 5385
  ↳ samples seen:   171930
Epoch 15 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.58it/s]
Epoch 15 → Eval accuracy: 0.7907


## 2

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:03<00:00,  1.29it/s]
Epoch 1 Training: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]

Epoch 1 avg training loss: 1.1259
  ↳ gradient steps: 1
  ↳ samples seen:   2
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 17.95it/s]
Epoch 1 → Eval accuracy: 0.5834

Epoch 2 Training: 100%|██████████| 1/1 [00:00<00:00,  3.85it/s]

Epoch 2 avg training loss: 0.8701
  ↳ gradient steps: 2
  ↳ samples seen:   4
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:17<00:00, 19.13it/s]
Epoch 2 → Eval accuracy: 0.6205

Epoch 3 Training: 100%|██████████| 1/1 [00:00<00:00,  3.99it/s]

Epoch 3 avg training loss: 0.6064
  ↳ gradient steps: 3
  ↳ samples seen:   6
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:09<00:00, 21.45it/s]
Epoch 3 → Eval accuracy: 0.6144

Epoch 4 Training: 100%|██████████| 1/1 [00:00<00:00,  3.98it/s]

Epoch 4 avg training loss: 0.4550
  ↳ gradient steps: 4
  ↳ samples seen:   8
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.23it/s]
Epoch 4 → Eval accuracy: 0.5861

Epoch 5 Training: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]

Epoch 5 avg training loss: 0.4229
  ↳ gradient steps: 5
  ↳ samples seen:   10
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:06<00:00, 22.46it/s]
Epoch 5 → Eval accuracy: 0.5868

Epoch 6 Training: 100%|██████████| 1/1 [00:00<00:00,  4.05it/s]

Epoch 6 avg training loss: 0.3636
  ↳ gradient steps: 6
  ↳ samples seen:   12
Epoch 6 Eval: 100%|██████████| 1486/1486 [00:46<00:00, 31.88it/s]
Epoch 6 → Eval accuracy: 0.5983

Epoch 7 Training: 100%|██████████| 1/1 [00:00<00:00,  4.98it/s]

Epoch 7 avg training loss: 0.2934
  ↳ gradient steps: 7
  ↳ samples seen:   14
Epoch 7 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.46it/s]
Epoch 7 → Eval accuracy: 0.6306

Epoch 8 Training: 100%|██████████| 1/1 [00:00<00:00,  5.04it/s]

Epoch 8 avg training loss: 0.2187
  ↳ gradient steps: 8
  ↳ samples seen:   16
Epoch 8 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.40it/s]
Epoch 8 → Eval accuracy: 0.6763

Epoch 9 Training: 100%|██████████| 1/1 [00:00<00:00,  4.91it/s]

Epoch 9 avg training loss: 0.1678
  ↳ gradient steps: 9
  ↳ samples seen:   18
Epoch 9 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.45it/s]
Epoch 9 → Eval accuracy: 0.7086

Epoch 10 Training: 100%|██████████| 1/1 [00:00<00:00,  4.88it/s]

Epoch 10 avg training loss: 0.1220
  ↳ gradient steps: 10
  ↳ samples seen:   20
Epoch 10 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.35it/s]
Epoch 10 → Eval accuracy: 0.7059

Epoch 11 Training: 100%|██████████| 1/1 [00:00<00:00,  4.93it/s]

Epoch 11 avg training loss: 0.0842
  ↳ gradient steps: 11
  ↳ samples seen:   22
Epoch 11 Eval: 100%|██████████| 1486/1486 [00:44<00:00, 33.11it/s]
Epoch 11 → Eval accuracy: 0.6817

Epoch 12 Training: 100%|██████████| 1/1 [00:00<00:00,  4.52it/s]

Epoch 12 avg training loss: 0.0614
  ↳ gradient steps: 12
  ↳ samples seen:   24
Epoch 12 Eval: 100%|██████████| 1486/1486 [00:48<00:00, 30.73it/s]
Epoch 12 → Eval accuracy: 0.6649

Epoch 13 Training: 100%|██████████| 1/1 [00:00<00:00,  5.10it/s]

Epoch 13 avg training loss: 0.0433
  ↳ gradient steps: 13
  ↳ samples seen:   26
Epoch 13 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.52it/s]
Epoch 13 → Eval accuracy: 0.6514

Epoch 14 Training: 100%|██████████| 1/1 [00:00<00:00,  5.20it/s]

Epoch 14 avg training loss: 0.0298
  ↳ gradient steps: 14
  ↳ samples seen:   28
Epoch 14 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.47it/s]
Epoch 14 → Eval accuracy: 0.6393

Epoch 15 Training: 100%|██████████| 1/1 [00:00<00:00,  5.16it/s]

Epoch 15 avg training loss: 0.0202
  ↳ gradient steps: 15
  ↳ samples seen:   30
Epoch 15 Eval: 100%|██████████| 1486/1486 [00:43<00:00, 34.54it/s]
Epoch 15 → Eval accuracy: 0.6285

Epoch 16 Training: 100%|██████████| 1/1 [00:00<00:00,  5.27it/s]

Epoch 16 avg training loss: 0.0131
  ↳ gradient steps: 16
  ↳ samples seen:   32

## 4

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:03<00:00,  1.24it/s]
Epoch 1 Training: 100%|██████████| 1/1 [00:02<00:00,  2.30s/it]

Epoch 1 avg training loss: 1.0515
  ↳ gradient steps: 1
  ↳ samples seen:   4
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 17.97it/s]
Epoch 1 → Eval accuracy: 0.5875

Epoch 2 Training: 100%|██████████| 1/1 [00:00<00:00,  3.30it/s]

Epoch 2 avg training loss: 0.8224
  ↳ gradient steps: 2
  ↳ samples seen:   8
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.29it/s]
Epoch 2 → Eval accuracy: 0.6003

Epoch 3 Training: 100%|██████████| 1/1 [00:00<00:00,  3.68it/s]

Epoch 3 avg training loss: 0.6402
  ↳ gradient steps: 3
  ↳ samples seen:   12
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.46it/s]
Epoch 3 → Eval accuracy: 0.5673

Epoch 4 Training: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]

Epoch 4 avg training loss: 0.5792
  ↳ gradient steps: 4
  ↳ samples seen:   16
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.35it/s]
Epoch 4 → Eval accuracy: 0.5444

Epoch 5 Training: 100%|██████████| 1/1 [00:00<00:00,  3.68it/s]

Epoch 5 avg training loss: 0.5420
  ↳ gradient steps: 5
  ↳ samples seen:   20


## 8

In [ ]:
Epoch 1 Training: 100%|██████████| 1/1 [00:02<00:00,  2.43s/it]

Epoch 1 avg training loss: 0.9227
  ↳ gradient steps: 1
  ↳ samples seen:   8
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:18<00:00, 18.99it/s]
Epoch 1 → Eval accuracy: 0.5875

Epoch 2 Training: 100%|██████████| 1/1 [00:00<00:00,  3.36it/s]

Epoch 2 avg training loss: 0.7603
  ↳ gradient steps: 2
  ↳ samples seen:   16
Epoch 2 Eval: 100%|██████████| 1486/1486 [00:50<00:00, 29.46it/s]
Epoch 2 → Eval accuracy: 0.5821

Epoch 3 Training: 100%|██████████| 1/1 [00:00<00:00,  3.90it/s]

Epoch 3 avg training loss: 0.6537
  ↳ gradient steps: 3
  ↳ samples seen:   24
Epoch 3 Eval: 100%|██████████| 1486/1486 [00:42<00:00, 34.89it/s]
Epoch 3 → Eval accuracy: 0.5444

Epoch 4 Training: 100%|██████████| 1/1 [00:00<00:00,  3.94it/s]

Epoch 4 avg training loss: 0.6588
  ↳ gradient steps: 4
  ↳ samples seen:   32
Epoch 4 Eval: 100%|██████████| 1486/1486 [00:44<00:00, 33.40it/s]
Epoch 4 → Eval accuracy: 0.5330

Epoch 5 Training: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]

Epoch 5 avg training loss: 0.6493
  ↳ gradient steps: 5
  ↳ samples seen:   40
Epoch 5 Eval: 100%|██████████| 1486/1486 [00:42<00:00, 35.18it/s]
Epoch 5 → Eval accuracy: 0.5350

Epoch 6 Training: 100%|██████████| 1/1 [00:00<00:00,  3.96it/s]

Epoch 6 avg training loss: 0.6126
  ↳ gradient steps: 6
  ↳ samples seen:   48
Epoch 6 Eval: 100%|██████████| 1486/1486 [00:49<00:00, 29.72it/s]
Epoch 6 → Eval accuracy: 0.5498

Epoch 7 Training: 100%|██████████| 1/1 [00:00<00:00,  3.56it/s]

Epoch 7 avg training loss: 0.5630
  ↳ gradient steps: 7
  ↳ samples seen:   56
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:08<00:00, 21.63it/s]
Epoch 7 → Eval accuracy: 0.5841

Epoch 8 Training: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]

Epoch 8 avg training loss: 0.5164
  ↳ gradient steps: 8
  ↳ samples seen:   64


## 256

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:03<00:00,  1.30it/s]
Epoch 1 Training: 100%|██████████| 8/8 [00:12<00:00,  1.57s/it]

Epoch 1 avg training loss: 0.6744
  ↳ gradient steps: 8
  ↳ samples seen:   256
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:18<00:00, 18.87it/s]
Epoch 1 → Eval accuracy: 0.6521

Epoch 2 Training: 100%|██████████| 8/8 [00:10<00:00,  1.32s/it]

Epoch 2 avg training loss: 0.5615
  ↳ gradient steps: 16
  ↳ samples seen:   512
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.34it/s]
Epoch 2 → Eval accuracy: 0.7268

Epoch 3 Training: 100%|██████████| 8/8 [00:11<00:00,  1.42s/it]

Epoch 3 avg training loss: 0.5005
  ↳ gradient steps: 24
  ↳ samples seen:   768
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:18<00:00, 18.99it/s]
Epoch 3 → Eval accuracy: 0.7301

Epoch 4 Training: 100%|██████████| 8/8 [00:11<00:00,  1.39s/it]

Epoch 4 avg training loss: 0.4421
  ↳ gradient steps: 32
  ↳ samples seen:   1024
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.45it/s]
Epoch 4 → Eval accuracy: 0.6817

Epoch 5 Training: 100%|██████████| 8/8 [00:10<00:00,  1.34s/it]

Epoch 5 avg training loss: 0.3439
  ↳ gradient steps: 40
  ↳ samples seen:   1280
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:12<00:00, 20.52it/s]
Epoch 5 → Eval accuracy: 0.6696

Epoch 6 Training: 100%|██████████| 8/8 [00:11<00:00,  1.41s/it]

Epoch 6 avg training loss: 0.2779
  ↳ gradient steps: 48
  ↳ samples seen:   1536
Epoch 6 Eval: 100%|██████████| 1486/1486 [00:42<00:00, 34.76it/s]
Epoch 6 → Eval accuracy: 0.7517

Epoch 7 Training: 100%|██████████| 8/8 [00:10<00:00,  1.30s/it]

Epoch 7 avg training loss: 0.1592
  ↳ gradient steps: 56
  ↳ samples seen:   1792
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:11<00:00, 20.70it/s]
Epoch 7 → Eval accuracy: 0.7530

Epoch 8 Training: 100%|██████████| 8/8 [00:11<00:00,  1.45s/it]

Epoch 8 avg training loss: 0.0646
  ↳ gradient steps: 64
  ↳ samples seen:   2048
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.34it/s]
Epoch 8 → Eval accuracy: 0.7052

Epoch 9 Training: 100%|██████████| 8/8 [00:11<00:00,  1.38s/it]

Epoch 9 avg training loss: 0.0330
  ↳ gradient steps: 72
  ↳ samples seen:   2304
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.54it/s]
Epoch 9 → Eval accuracy: 0.7638

Epoch 10 Training: 100%|██████████| 8/8 [00:11<00:00,  1.45s/it]

Epoch 10 avg training loss: 0.0221
  ↳ gradient steps: 80
  ↳ samples seen:   2560


## 32

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:43<00:00,  9.41s/it]
Epoch 1 Training: 100%|██████████| 1/1 [00:02<00:00,  2.85s/it]

Epoch 1 avg training loss: 0.8034
  ↳ gradient steps: 1
  ↳ samples seen:   32
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.34it/s]
Epoch 1 → Eval accuracy: 0.5915

Epoch 2 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 2 avg training loss: 0.6931
  ↳ gradient steps: 2
  ↳ samples seen:   64
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:06<00:00, 22.40it/s]
Epoch 2 → Eval accuracy: 0.5794

Epoch 3 Training: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Epoch 3 avg training loss: 0.6386
  ↳ gradient steps: 3
  ↳ samples seen:   96
Epoch 3 Eval: 100%|██████████| 1486/1486 [00:47<00:00, 31.38it/s]
Epoch 3 → Eval accuracy: 0.5491

Epoch 4 Training: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Epoch 4 avg training loss: 0.6469
  ↳ gradient steps: 4
  ↳ samples seen:   128
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:07<00:00, 22.04it/s]
Epoch 4 → Eval accuracy: 0.5471

Epoch 5 Training: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Epoch 5 avg training loss: 0.6353
  ↳ gradient steps: 5
  ↳ samples seen:   160
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.53it/s]
Epoch 5 → Eval accuracy: 0.5619

Epoch 6 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 6 avg training loss: 0.5976
  ↳ gradient steps: 6
  ↳ samples seen:   192
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.19it/s]
Epoch 6 → Eval accuracy: 0.5949

Epoch 7 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 7 avg training loss: 0.5625
  ↳ gradient steps: 7
  ↳ samples seen:   224
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.39it/s]
Epoch 7 → Eval accuracy: 0.6238

Epoch 8 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 8 avg training loss: 0.5290
  ↳ gradient steps: 8
  ↳ samples seen:   256
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.14it/s]
Epoch 8 → Eval accuracy: 0.6487

Epoch 9 Training: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Epoch 9 avg training loss: 0.5091
  ↳ gradient steps: 9
  ↳ samples seen:   288
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:13<00:00, 20.08it/s]
Epoch 9 → Eval accuracy: 0.6736

Epoch 10 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 10 avg training loss: 0.4873
  ↳ gradient steps: 10
  ↳ samples seen:   320
Epoch 10 Eval: 100%|██████████| 1486/1486 [01:14<00:00, 19.85it/s]
Epoch 10 → Eval accuracy: 0.6830

Epoch 11 Training: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Epoch 11 avg training loss: 0.4584
  ↳ gradient steps: 11
  ↳ samples seen:   352
Epoch 11 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.56it/s]
Epoch 11 → Eval accuracy: 0.6864

Epoch 12 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 12 avg training loss: 0.4364
  ↳ gradient steps: 12
  ↳ samples seen:   384
Epoch 12 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.23it/s]
Epoch 12 → Eval accuracy: 0.6945

Epoch 13 Training: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]

Epoch 13 avg training loss: 0.3978
  ↳ gradient steps: 13
  ↳ samples seen:   416
Epoch 13 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.01it/s]
Epoch 13 → Eval accuracy: 0.6952

Epoch 14 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 14 avg training loss: 0.3620
  ↳ gradient steps: 14
  ↳ samples seen:   448
Epoch 14 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.22it/s]
Epoch 14 → Eval accuracy: 0.6904

Epoch 15 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 15 avg training loss: 0.3228
  ↳ gradient steps: 15
  ↳ samples seen:   480
Epoch 16 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.40it/s]
Epoch 16 → Eval accuracy: 0.6763

Epoch 17 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 17 avg training loss: 0.2408
  ↳ gradient steps: 17
  ↳ samples seen:   544
Epoch 17 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.34it/s]
Epoch 17 → Eval accuracy: 0.6736

Epoch 18 Training: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]

Epoch 18 avg training loss: 0.1961
  ↳ gradient steps: 18
  ↳ samples seen:   576


## 4096

In [ ]:
Epoch 1 Training: 100%|██████████| 128/128 [02:50<00:00,  1.33s/it]

Epoch 1 avg training loss: 0.5058
  ↳ gradient steps: 128
  ↳ samples seen:   4096
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.03it/s]
Epoch 1 → Eval accuracy: 0.7826

  ↳ gradient steps: 128
  ↳ samples seen:   4096
Epoch 2 Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 2 avg training loss: 0.3868
  ↳ gradient steps: 256
  ↳ samples seen:   8192
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.31it/s]
Epoch 2 → Eval accuracy: 0.8069

  ↳ gradient steps: 256
  ↳ samples seen:   8192
Epoch 3 Training: 100%|██████████| 128/128 [02:49<00:00,  1.32s/it]

Epoch 3 avg training loss: 0.3060
  ↳ gradient steps: 384
  ↳ samples seen:   12288
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:09<00:00, 21.47it/s]
Epoch 3 → Eval accuracy: 0.7672

  ↳ gradient steps: 384
  ↳ samples seen:   12288
Epoch 4 Training: 100%|██████████| 128/128 [02:50<00:00,  1.33s/it]

Epoch 4 avg training loss: 0.2257
  ↳ gradient steps: 512
  ↳ samples seen:   16384
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.41it/s]
Epoch 4 → Eval accuracy: 0.7941

  ↳ gradient steps: 512
  ↳ samples seen:   16384
Epoch 5 Training: 100%|██████████| 128/128 [02:51<00:00,  1.34s/it]

Epoch 5 avg training loss: 0.1515
  ↳ gradient steps: 640
  ↳ samples seen:   20480
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.33it/s]
Epoch 5 → Eval accuracy: 0.8069

  ↳ gradient steps: 640
  ↳ samples seen:   20480
Epoch 6 Training: 100%|██████████| 128/128 [02:45<00:00,  1.29s/it]

Epoch 6 avg training loss: 0.1109
  ↳ gradient steps: 768
  ↳ samples seen:   24576
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:16<00:00, 19.36it/s]
Epoch 6 → Eval accuracy: 0.8109

  ↳ gradient steps: 768
  ↳ samples seen:   24576
Epoch 7 Training: 100%|██████████| 128/128 [02:51<00:00,  1.34s/it]

Epoch 7 avg training loss: 0.0599
  ↳ gradient steps: 896
  ↳ samples seen:   28672
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:02<00:00, 23.81it/s]
Epoch 7 → Eval accuracy: 0.8324

  ↳ gradient steps: 896
  ↳ samples seen:   28672
Epoch 8 Training: 100%|██████████| 128/128 [02:49<00:00,  1.32s/it]

Epoch 8 avg training loss: 0.0445
  ↳ gradient steps: 1024
  ↳ samples seen:   32768
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:07<00:00, 22.03it/s]
Epoch 8 → Eval accuracy: 0.8176

  ↳ gradient steps: 1024
  ↳ samples seen:   32768
Epoch 9 Training: 100%|██████████| 128/128 [02:47<00:00,  1.31s/it]

Epoch 9 avg training loss: 0.0498
  ↳ gradient steps: 1152
  ↳ samples seen:   36864
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.49it/s]
Epoch 9 → Eval accuracy: 0.8244

  ↳ gradient steps: 1152
  ↳ samples seen:   36864

## 16384

In [ ]:
Epoch 0 Eval: 100%|██████████| 1486/1486 [01:23<00:00, 17.77it/s]
Epoch 0 → Eval accuracy: 0.5518

Epoch 1 Training: 100%|██████████| 359/359 [07:15<00:00,  1.21s/it]

Epoch 1 avg training loss: 0.4362
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.25it/s]
Epoch 1 → Eval accuracy: 0.7981

Epoch 2 Training: 100%|██████████| 359/359 [07:17<00:00,  1.22s/it]

Epoch 2 avg training loss: 0.3199
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:22<00:00, 18.04it/s]
Epoch 2 → Eval accuracy: 0.8560

Epoch 3 Training: 100%|██████████| 359/359 [07:18<00:00,  1.22s/it]

Epoch 3 avg training loss: 0.2515
Epoch 3 Eval: 100%|██████████| 1486/1486 [00:59<00:00, 24.93it/s]
Epoch 3 → Eval accuracy: 0.8526

Epoch 4 Training: 100%|██████████| 359/359 [07:22<00:00,  1.23s/it]

Epoch 4 avg training loss: 0.1719
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.81it/s]
Epoch 4 → Eval accuracy: 0.8681

Epoch 5 Training: 100%|██████████| 359/359 [07:25<00:00,  1.24s/it]

Epoch 5 avg training loss: 0.0953
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:11<00:00, 20.72it/s]
Epoch 5 → Eval accuracy: 0.8493

Epoch 6 Training: 100%|██████████| 359/359 [07:21<00:00,  1.23s/it]

Epoch 6 avg training loss: 0.0529
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:12<00:00, 20.38it/s]
Epoch 6 → Eval accuracy: 0.8513

Epoch 7 Training: 100%|██████████| 359/359 [07:20<00:00,  1.23s/it]

Epoch 7 avg training loss: 0.0325
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.21it/s]
Epoch 7 → Eval accuracy: 0.8600

Epoch 8 Training: 100%|██████████| 359/359 [07:17<00:00,  1.22s/it]

Epoch 8 avg training loss: 0.0244
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.33it/s]
Epoch 8 → Eval accuracy: 0.8587

Epoch 9 Training: 100%|██████████| 359/359 [07:20<00:00,  1.23s/it]

Epoch 9 avg training loss: 0.0206
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:15<00:00, 19.63it/s]
Epoch 9 → Eval accuracy: 0.8439

Epoch 10 Training: 100%|██████████| 359/359 [07:24<00:00,  1.24s/it]

Epoch 10 avg training loss: 0.0143


## 1024

In [ ]:
Loading checkpoint shards: 100%
 4/4 [00:03<00:00,  1.30it/s]
Epoch 0 Eval: 100%|██████████| 1486/1486 [01:19<00:00, 18.80it/s]
Epoch 0 → Eval accuracy: 0.5518

Epoch 1 Training: 100%|██████████| 32/32 [00:42<00:00,  1.33s/it]

Epoch 1 avg training loss: 0.6267
Epoch 1 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.51it/s]
Epoch 1 → Eval accuracy: 0.7456

Epoch 2 Training: 100%|██████████| 32/32 [00:41<00:00,  1.30s/it]

Epoch 2 avg training loss: 0.4780
Epoch 2 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.26it/s]
Epoch 2 → Eval accuracy: 0.7510

Epoch 3 Training: 100%|██████████| 32/32 [00:38<00:00,  1.20s/it]

Epoch 3 avg training loss: 0.3747
Epoch 3 Eval: 100%|██████████| 1486/1486 [01:02<00:00, 23.74it/s]
Epoch 3 → Eval accuracy: 0.7550

Epoch 4 Training: 100%|██████████| 32/32 [00:41<00:00,  1.30s/it]

Epoch 4 avg training loss: 0.2698
Epoch 4 Eval: 100%|██████████| 1486/1486 [01:19<00:00, 18.76it/s]
Epoch 4 → Eval accuracy: 0.7376

Epoch 5 Training: 100%|██████████| 32/32 [00:40<00:00,  1.26s/it]

Epoch 5 avg training loss: 0.2338
Epoch 5 Eval: 100%|██████████| 1486/1486 [01:12<00:00, 20.38it/s]
Epoch 5 → Eval accuracy: 0.7510

Epoch 6 Training: 100%|██████████| 32/32 [00:40<00:00,  1.26s/it]

Epoch 6 avg training loss: 0.1708
Epoch 6 Eval: 100%|██████████| 1486/1486 [01:04<00:00, 23.14it/s]
Epoch 6 → Eval accuracy: 0.7355

Epoch 7 Training: 100%|██████████| 32/32 [00:40<00:00,  1.27s/it]

Epoch 7 avg training loss: 0.1069
Epoch 7 Eval: 100%|██████████| 1486/1486 [01:09<00:00, 21.24it/s]
Epoch 7 → Eval accuracy: 0.7591

Epoch 8 Training: 100%|██████████| 32/32 [00:41<00:00,  1.29s/it]

Epoch 8 avg training loss: 0.0540
Epoch 8 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.45it/s]
Epoch 8 → Eval accuracy: 0.7611

Epoch 9 Training: 100%|██████████| 32/32 [00:40<00:00,  1.26s/it]

Epoch 9 avg training loss: 0.0335
Epoch 9 Eval: 100%|██████████| 1486/1486 [01:11<00:00, 20.66it/s]
Epoch 9 → Eval accuracy: 0.7476

Epoch 10 Training: 100%|██████████| 32/32 [00:40<00:00,  1.27s/it]

Epoch 10 avg training loss: 0.0121
Epoch 10 Eval: 100%|██████████| 1486/1486 [01:13<00:00, 20.14it/s]
Epoch 10 → Eval accuracy: 0.7409

Epoch 11 Training: 100%|██████████| 32/32 [00:38<00:00,  1.20s/it]

Epoch 11 avg training loss: 0.0054
Epoch 11 Eval: 100%|██████████| 1486/1486 [01:14<00:00, 20.00it/s]
Epoch 11 → Eval accuracy: 0.7699

Epoch 12 Training: 100%|██████████| 32/32 [00:39<00:00,  1.24s/it]

Epoch 12 avg training loss: 0.0043
Epoch 12 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.49it/s]
Epoch 12 → Eval accuracy: 0.7416

Epoch 13 Training: 100%|██████████| 32/32 [00:38<00:00,  1.22s/it]

Epoch 13 avg training loss: 0.0014
Epoch 13 Eval: 100%|██████████| 1486/1486 [01:11<00:00, 20.85it/s]
Epoch 13 → Eval accuracy: 0.7503

Epoch 14 Training: 100%|██████████| 32/32 [00:40<00:00,  1.25s/it]

Epoch 14 avg training loss: 0.0030
Epoch 14 Eval: 100%|██████████| 1486/1486 [01:21<00:00, 18.17it/s]
Epoch 14 → Eval accuracy: 0.7530

Epoch 15 Training: 100%|██████████| 32/32 [00:41<00:00,  1.30s/it]

Epoch 15 avg training loss: 0.0002
Epoch 15 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.46it/s]
Epoch 15 → Eval accuracy: 0.7624

Epoch 16 Training: 100%|██████████| 32/32 [00:40<00:00,  1.27s/it]

Epoch 16 avg training loss: 0.0002
Epoch 16 Eval: 100%|██████████| 1486/1486 [01:20<00:00, 18.49it/s]
Epoch 16 → Eval accuracy: 0.7584

Epoch 17 Training: 100%|██████████| 32/32 [00:40<00:00,  1.26s/it]

Epoch 17 avg training loss: 0.0001


In [3]:
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from accelerate import Accelerator
import matplotlib.pyplot as plt

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME      = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"
TRAIN_PATH      = "data/train_gsm8k_preference.json"
TEST_PATH       = "data/test_gsm8k_preference.json"
TRAIN_SUBSET    = 1024       # use the first 256 of train for fine‑tuning
LR              = 1e-4       # high LR for strong effect
RANK            = 128        # LoRA rank    
BATCH_SIZE      = 4          # number of pairs per gradient‐step

# ── SETUP ACCELERATOR ──────────────────────────────────────────────────────────
accelerator = Accelerator(mixed_precision="bf16")
device      = accelerator.device

# ── TOKENIZER & MODEL ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── LO­RA WRAPPER ──────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=RANK,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)

# ── TRUE/FALSE TOKEN IDS ───────────────────────────────────────────────────────
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0] 
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]

# ── PROMPT TEMPLATE ───────────────────────────────────────────────────────────
template = (
    "Question: {question}",
    "Claim: {answer}",
    "I think this claim is"
)

# ── LOAD TRAIN SET ─────────────────────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    train_data = json.load(f)
train_data = train_data[:TRAIN_SUBSET]

train_qs      = [ex["question"]       for ex in train_data]
train_c1      = [ex["choice"]         for ex in train_data]
train_c2      = [ex["choice_2"]       for ex in train_data]
train_labels  = [ex["label"].lower()  for ex in train_data]
train_cids    = [ex["consistency_id"] for ex in train_data]

# group train into pairs
groups = defaultdict(list)
for i, cid in enumerate(train_cids):
    groups[cid].append(i)
train_pairs = [grp for grp in groups.values() if len(grp) == 2]

# build train prompts
train_prompts = [
    template.format(question=train_qs[i], c1=train_c1[i], c2=train_c2[i])
    for i in range(len(train_data))
]

# ── LOAD TEST SET ──────────────────────────────────────────────────────────────
with open(TEST_PATH) as f:
    test_data = json.load(f)

test_qs     = [ex["question"]       for ex in test_data]
test_c1     = [ex["choice"]         for ex in test_data]
test_c2     = [ex["choice_2"]       for ex in test_data]
test_labels = [ex["label"].lower()  for ex in test_data]
test_cids   = [ex["consistency_id"] for ex in test_data]

# group test into pairs
groups = defaultdict(list)
for i, cid in enumerate(test_cids):
    groups[cid].append(i)
test_pairs = [grp for grp in groups.values() if len(grp) == 2]

# build test prompts
test_prompts = [
    template.format(question=test_qs[i], c1=test_c1[i], c2=test_c2[i])
    for i in range(len(test_data))
]

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
@torch.no_grad()
def score_text(prompt: str) -> float:
    enc = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=4096)
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc).logits                  # [1, seq_len, vocab]
    seq_len = enc["attention_mask"].sum(dim=1)    # [1]
    last = logits[0, seq_len-1]                   # [vocab]
    logp = torch.log_softmax(last, dim=-1)
    return (logp[true_id] - logp[false_id]).item()

# ── BATCH SCORING FN ───────────────────────────────────────────────────────────
@torch.no_grad()
def score_pair(p0: str, p1: str) -> (float, float):
    enc = tokenizer([p0, p1], return_tensors="pt", padding=True, truncation=True, max_length=4096)
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc).logits                  # [2, seq_len, vocab]
    lens = enc["attention_mask"].sum(dim=1)       # [2]
    last = logits[torch.arange(2), lens-1]        # [2, vocab]
    logp = torch.log_softmax(last, dim=-1)
    return (logp[:, true_id] - logp[:, false_id]).cpu().tolist()

# ── ZERO‑SHOT EVAL ON TEST ─────────────────────────────────────────────────────
print("== Zero‑Shot Evaluation on Test Set ==")
# correct = total = 0
# for a, b in tqdm(test_pairs, desc="Zero‑Shot Test"):
#     p_a, p_b = test_prompts[a], test_prompts[b]
#     d_a, d_b = score_pair(p_a, p_b)
#     # assign predictions
#     if d_a >= 0:
#         pred_a = "true"
#     else:
#         pred_a = "false"
        
#     if d_b >= 0:
#         pred_b = "true"
#     else:
#         pred_b = "false"
#     # tally
#     correct += (pred_a == test_labels[a]) + (pred_b == test_labels[b])
#     total   += 2
# print(f"Zero‑Shot Test Accuracy: {correct/total:.4f}\n")

# accuracy = correct / total

# ── PREPARE FOR FINE‑TUNING ────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model, optimizer = accelerator.prepare(model, optimizer)

# ── FINE‑TUNE ON TRAIN SET WITH PSEUDO‑LABELS ────────────────────────────────
accuracies = []
print("== Fine‑Tuning on Train Set ==")
pair_loader = DataLoader(
    train_pairs,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda x: x,   # x is already a list of (i,j) pairs
)

# 2) infinite training loop with mini-batches of size=BATCH_SIZE
while True:
    model.train()
    batch_idx = 0
    for batch_pairs in tqdm(pair_loader, desc="Fine‑tuning batches"):
        # 1) Build your prompts_flat_learn (no demo prefix) for ground‑truth training
        model.train()
        prompts_flat_learn = []
        gold_indices       = []
        for (i, j) in batch_pairs:
            prompts_flat_learn.append(train_prompts[i])
            prompts_flat_learn.append(train_prompts[j])
            # record the true/false label for each sample
            gold_indices.append(1 if train_labels[i] == "true"  else 0)
            gold_indices.append(1 if train_labels[j] == "true"  else 0)

        # 2) Tokenize & forward
        enc2 = tokenizer(
            prompts_flat_learn,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096,
        ).to(device)
        logits2  = model(**enc2).logits                                    # [2*BATCH, seq_len, vocab]
        lengths2 = enc2["attention_mask"].sum(dim=1)                        # [2*BATCH]
        last2    = logits2[torch.arange(len(prompts_flat_learn)), lengths2-1]  # [2*BATCH, vocab]

        # 3) Extract the two token logits
        l_true2      = last2[:, true_id]                                    # [2*BATCH]
        l_false2     = last2[:, false_id]                                   # [2*BATCH]
        pair_logits2 = torch.stack([l_false2, l_true2], dim=1)              # [2*BATCH, 2]

        # 4) Build your gold Tensor from the real labels
        gold = torch.tensor(gold_indices, device=device)                    # values in {0,1}

        # 5) Compute loss & step
        loss = F.cross_entropy(pair_logits2, gold)
        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        batch_idx += 1
        # if batch_idx*BATCH_SIZE in [32, 128, 256, 512]:
            # ── EVAL ON TEST AFTER FINE‑TUNING ─────────────────────────────────────────────
    print("\n== Post‑Fine‑Tuning Evaluation on Test Set ==")
    model.eval()
    correct = total = 0

    # replace this:
    # for a, b in tqdm(test_pairs, desc="Post‑Fine‑Tune Test"):

    # with a plain for‑loop:
    # for a, b in test_pairs:
    #     p_a, p_b = test_prompts[a], test_prompts[b]
    #     d_a, d_b = score_pair(p_a, p_b)
    #     pred_a = "true" if d_a >= 0 else "false"
    #     pred_b = "true" if d_b >= 0 else "false"
    #     correct += (pred_a == test_labels[a]) + (pred_b == test_labels[b])
    #     total   += 2

    # print(f"Fine‑Tuned Test Accuracy after {batch_idx*BATCH_SIZE} pairs: {correct/total:.4f}")
            
    model.eval()
    correct = total = 0
    for a, b in tqdm(test_pairs, desc="Post‑Fine‑Tune Test"):
        p_a, p_b = test_prompts[a], test_prompts[b]
        d_a, d_b = score_pair(p_a, p_b)
        if d_a >= 0:   
            pred_a = "true"
        else:
            pred_a = "false"
            
        if d_b >= 0:  
            pred_b = "true"
        else:
            pred_b = "false"
        correct += (pred_a == test_labels[a]) + (pred_b == test_labels[b])
        total   += 2 
        
    print(f"Fine‑Tuned Test Accuracy: {correct/total:.4f}")

# ── OPTIONAL: PLOT ACCURACY ────────────────────────────────────────────────────
# plt.bar(["Zero‑Shot", "Fine‑Tuned"], [zero_shot_acc, fine_tuned_acc])
# plt.ylabel("Accuracy"); plt.show()

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

== Zero‑Shot Evaluation on Test Set ==
== Fine‑Tuning on Train Set ==


Fine‑tuning batches: 100%|██████████| 128/128 [04:04<00:00,  1.91s/it]



== Post‑Fine‑Tuning Evaluation on Test Set ==


Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.63it/s]


...


== Post‑Fine‑Tuning Evaluation on Test Set ==


Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.65it/s]


Fine‑Tuned Test Accuracy: 0.6349


Fine‑tuning batches:  61%|██████    | 78/128 [02:28<01:35,  1.90s/it]


KeyboardInterrupt: 

## Zero Shot

In [ ]:
== Zero‑Shot Evaluation on Test Set ==
Zero‑Shot Test: 100%|██████████| 467/467 [01:27<00:00,  5.35it/s]
Zero‑Shot Test Accuracy: 0.5450

== Fine‑Tuning on Train Set ==
Fine‑tuning batches:   5%|▌         | 7/128 [00:13<03:52,  1.92s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:   6%|▋         | 8/128 [01:35<54:50, 27.42s/it]
Fine‑Tuned Test Accuracy after 32 pairs: 0.5664
Fine‑tuning batches:  24%|██▍       | 31/128 [02:15<02:47,  1.73s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  25%|██▌       | 32/128 [03:38<41:39, 26.04s/it]
Fine‑Tuned Test Accuracy after 128 pairs: 0.5482
Fine‑tuning batches:  49%|████▉     | 63/128 [04:41<01:57,  1.81s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  50%|█████     | 64/128 [06:06<28:28, 26.70s/it]
Fine‑Tuned Test Accuracy after 256 pairs: 0.5567
Fine‑tuning batches:  99%|█████████▉| 127/128 [07:57<00:01,  1.23s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches: 100%|██████████| 128/128 [09:19<00:00,  4.37s/it]
Fine‑Tuned Test Accuracy after 512 pairs: 0.5792
Fine‑tuning batches:   5%|▌         | 7/128 [00:11<03:21,  1.66s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:   6%|▋         | 8/128 [01:32<53:13, 26.61s/it]
Fine‑Tuned Test Accuracy after 32 pairs: 0.5889
Fine‑tuning batches:  24%|██▍       | 31/128 [02:17<03:30,  2.17s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  25%|██▌       | 32/128 [03:40<42:29, 26.56s/it]
Fine‑Tuned Test Accuracy after 128 pairs: 0.6081
Fine‑tuning batches:  49%|████▉     | 63/128 [04:27<01:15,  1.16s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  50%|█████     | 64/128 [05:52<27:56, 26.20s/it]
Fine‑Tuned Test Accuracy after 256 pairs: 0.6253
Fine‑tuning batches:  99%|█████████▉| 127/128 [07:54<00:01,  1.69s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches: 100%|██████████| 128/128 [09:15<00:00,  4.34s/it]
Fine‑Tuned Test Accuracy after 512 pairs: 0.6424
Fine‑tuning batches:   5%|▌         | 7/128 [00:11<03:16,  1.63s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:   6%|▋         | 8/128 [01:29<52:10, 26.09s/it]
Fine‑Tuned Test Accuracy after 32 pairs: 0.6574
Fine‑tuning batches:  24%|██▍       | 31/128 [02:12<02:23,  1.48s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  25%|██▌       | 32/128 [03:37<42:10, 26.36s/it]
Fine‑Tuned Test Accuracy after 128 pairs: 0.6595
Fine‑tuning batches:  49%|████▉     | 63/128 [04:30<01:59,  1.83s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  50%|█████     | 64/128 [05:55<28:29, 26.70s/it]
Fine‑Tuned Test Accuracy after 256 pairs: 0.6531
Fine‑tuning batches:  99%|█████████▉| 127/128 [07:51<00:02,  2.16s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches: 100%|██████████| 128/128 [09:12<00:00,  4.32s/it]
Fine‑Tuned Test Accuracy after 512 pairs: 0.6756
Fine‑tuning batches:   5%|▌         | 7/128 [00:11<03:25,  1.70s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:   6%|▋         | 8/128 [01:30<52:45, 26.38s/it]
Fine‑Tuned Test Accuracy after 32 pairs: 0.6660
Fine‑tuning batches:  24%|██▍       | 31/128 [02:08<02:33,  1.58s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  25%|██▌       | 32/128 [03:29<40:28, 25.29s/it]
Fine‑Tuned Test Accuracy after 128 pairs: 0.6842
Fine‑tuning batches:  49%|████▉     | 63/128 [04:21<01:43,  1.59s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  50%|█████     | 64/128 [05:41<26:52, 25.20s/it]
Fine‑Tuned Test Accuracy after 256 pairs: 0.6745
Fine‑tuning batches:  99%|█████████▉| 127/128 [07:43<00:01,  1.79s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches: 100%|██████████| 128/128 [09:09<00:00,  4.29s/it]
Fine‑Tuned Test Accuracy after 512 pairs: 0.6692
Fine‑tuning batches:   5%|▌         | 7/128 [00:10<02:47,  1.38s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:   6%|▋         | 8/128 [01:33<54:43, 27.37s/it]
Fine‑Tuned Test Accuracy after 32 pairs: 0.6799
Fine‑tuning batches:  24%|██▍       | 31/128 [02:22<04:24,  2.72s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  25%|██▌       | 32/128 [03:46<43:12, 27.01s/it]
Fine‑Tuned Test Accuracy after 128 pairs: 0.6692
Fine‑tuning batches:  49%|████▉     | 63/128 [04:45<02:14,  2.07s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  50%|█████     | 64/128 [06:05<27:16, 25.57s/it]
Fine‑Tuned Test Accuracy after 256 pairs: 0.6617
Fine‑tuning batches:  99%|█████████▉| 127/128 [07:56<00:01,  1.76s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==

== Post‑Fine‑Tuning Evaluation on Test Set ==
Fine‑tuning batches:  99%|█████████▉| 127/128 [08:03<00:03,  3.81s/it]

## Chain Kuli Ki Main Kuli

In [ ]:
Loading checkpoint shards: 100%
 30/30 [00:32<00:00,  1.09it/s]
== Zero‑Shot Evaluation on Test Set ==
== Fine‑Tuning on Train Set ==
Fine‑tuning batches: 100%|██████████| 64/64 [02:22<00:00,  2.23s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.61it/s]
Fine‑Tuned Test Accuracy: 0.6670
Fine‑tuning batches: 100%|██████████| 64/64 [02:19<00:00,  2.17s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.59it/s]
Fine‑Tuned Test Accuracy: 0.6734
Fine‑tuning batches: 100%|██████████| 64/64 [02:19<00:00,  2.18s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6660
Fine‑tuning batches: 100%|██████████| 64/64 [02:19<00:00,  2.18s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:17<00:00,  6.02it/s]
Fine‑Tuned Test Accuracy: 0.6670
Fine‑tuning batches: 100%|██████████| 64/64 [02:20<00:00,  2.19s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.52it/s]
Fine‑Tuned Test Accuracy: 0.6681
Fine‑tuning batches: 100%|██████████| 64/64 [02:21<00:00,  2.21s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6670
Fine‑tuning batches: 100%|██████████| 64/64 [02:20<00:00,  2.19s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6702
Fine‑tuning batches: 100%|██████████| 64/64 [02:19<00:00,  2.19s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.64it/s]
Fine‑Tuned Test Accuracy: 0.6692
Fine‑tuning batches: 100%|██████████| 64/64 [02:19<00:00,  2.18s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.52it/s]
Fine‑Tuned Test Accuracy: 0.6692

In [ ]:
Loading checkpoint shards: 100%
 30/30 [00:33<00:00,  1.03it/s]
== Zero‑Shot Evaluation on Test Set ==
Zero‑Shot Test: 100%|██████████| 467/467 [01:27<00:00,  5.34it/s]
Zero‑Shot Test Accuracy: 0.5450

== Fine‑Tuning on Train Set ==
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.93s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.56it/s]
Fine‑Tuned Test Accuracy: 0.6381
Fine‑tuning batches: 100%|██████████| 16/16 [00:57<00:00,  3.57s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6445
Fine‑tuning batches: 100%|██████████| 16/16 [01:04<00:00,  4.01s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.51it/s]
Fine‑Tuned Test Accuracy: 0.6585
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.95s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.58it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.97s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6478
Fine‑tuning batches: 100%|██████████| 16/16 [01:01<00:00,  3.86s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.50it/s]
Fine‑Tuned Test Accuracy: 0.6734
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.90s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6702
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.78s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.63it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.75s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:17<00:00,  6.00it/s]
Fine‑Tuned Test Accuracy: 0.6767
Fine‑tuning batches: 100%|██████████| 16/16 [01:04<00:00,  4.04s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.56it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.76s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6809
Fine‑tuning batches: 100%|██████████| 16/16 [00:58<00:00,  3.63s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.78s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.57it/s]
Fine‑Tuned Test Accuracy: 0.6799
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.89s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:01<00:00,  3.85s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [00:57<00:00,  3.62s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6799
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.93s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.60it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.93s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.81s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [00:59<00:00,  3.71s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.52it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [00:58<00:00,  3.69s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.56it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.81s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.94s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.59it/s]
Fine‑Tuned Test Accuracy: 0.6767
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.88s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.51it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.88s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6799
Fine‑tuning batches: 100%|██████████| 16/16 [01:04<00:00,  4.04s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.63it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:04<00:00,  4.02s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.68it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [00:56<00:00,  3.54s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6745
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.97s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.52it/s]
Fine‑Tuned Test Accuracy: 0.6756
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.90s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6799
Fine‑tuning batches: 100%|██████████| 16/16 [00:59<00:00,  3.74s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.53it/s]
Fine‑Tuned Test Accuracy: 0.6745
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.90s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.94s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.61it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.75s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:23<00:00,  5.60it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.89s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:22<00:00,  5.64it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:01<00:00,  3.87s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.94s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:19<00:00,  5.85it/s]
Fine‑Tuned Test Accuracy: 0.6767
Fine‑tuning batches: 100%|██████████| 16/16 [01:03<00:00,  3.96s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.56it/s]
Fine‑Tuned Test Accuracy: 0.6799
Fine‑tuning batches: 100%|██████████| 16/16 [00:56<00:00,  3.54s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.54it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.89s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.55it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:00<00:00,  3.79s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.52it/s]
Fine‑Tuned Test Accuracy: 0.6777
Fine‑tuning batches: 100%|██████████| 16/16 [01:02<00:00,  3.90s/it]

== Post‑Fine‑Tuning Evaluation on Test Set ==
Post‑Fine‑Tune Test: 100%|██████████| 467/467 [01:24<00:00,  5.51it/s]
Fine‑Tuned Test Accuracy: 0.6788
Fine‑tuning batches:  75%|███████▌  | 12/16 [00:49<00:16,  4.11s/it]